<a href="https://colab.research.google.com/github/LEXEMEE/Python-Practice-Problems/blob/main/eduaid_project_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install openpyxl matplotlib fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=407d3f914e2983bc773b7fc8c7523a16d3dc80d083a95bfa22e9ef622d23c9e0
  Stored in directory: /root/.cache/pip/wheels/f9/95/ba/f418094659025eb9611f17cbcaf2334236bf39a0c3453ea455
Successfully built fpdf


In [2]:
import os
import openpyxl
from openpyxl import Workbook
import matplotlib.pyplot as plt
from fpdf import FPDF

BRANCHES_DIR = "branches"

os.makedirs(BRANCHES_DIR, exist_ok=True)


def create_branch(branch_name):
    """Create a new branch with an Excel database."""
    branch_path = os.path.join(BRANCHES_DIR, branch_name)
    os.makedirs(branch_path, exist_ok=True)
    excel_file = os.path.join(branch_path, "database.xlsx")

    if not os.path.exists(excel_file):
        workbook = Workbook()
        workbook.create_sheet(title="Students")
        workbook.create_sheet(title="Fees")
        workbook.create_sheet(title="Expenses")
        workbook.remove(workbook["Sheet"])
        workbook.save(excel_file)
        print(f"Branch '{branch_name}' created successfully.")
    else:
        print(f"Branch '{branch_name}' already exists.")


def list_branches():
    """List all available branches."""
    branches = os.listdir(BRANCHES_DIR)
    if not branches:
        print("No branches found.")
    else:
        print("Available branches:")
        for branch in branches:
            print(f"- {branch}")


def add_student(branch_name, student_id, name, grade):
    """Add a student to the branch database."""
    branch_path = os.path.join(BRANCHES_DIR, branch_name, "database.xlsx")
    if not os.path.exists(branch_path):
        print(f"Branch '{branch_name}' does not exist.")
        return

    workbook = openpyxl.load_workbook(branch_path)
    sheet = workbook["Students"]

    if sheet.max_row == 1:
        sheet.append(["Student ID", "Name", "Grade"])

    sheet.append([student_id, name, grade])
    workbook.save(branch_path)
    print(f"Student {name} added to branch '{branch_name}'.")


def record_fee(branch_name, student_id, amount):
    """Record a fee payment."""
    branch_path = os.path.join(BRANCHES_DIR, branch_name, "database.xlsx")
    if not os.path.exists(branch_path):
        print(f"Branch '{branch_name}' does not exist.")
        return

    workbook = openpyxl.load_workbook(branch_path)
    sheet = workbook["Fees"]

    if sheet.max_row == 1:
        sheet.append(["Student ID", "Amount", "Date"])

    sheet.append([student_id, amount, "2024-11-26"])  # Replace with dynamic date if needed
    workbook.save(branch_path)
    print(f"Fee of {amount} recorded for student ID {student_id} in branch '{branch_name}'.")


def record_expense(branch_name, description, amount):
    """Record an expense."""
    branch_path = os.path.join(BRANCHES_DIR, branch_name, "database.xlsx")
    if not os.path.exists(branch_path):
        print(f"Branch '{branch_name}' does not exist.")
        return

    workbook = openpyxl.load_workbook(branch_path)
    sheet = workbook["Expenses"]

    if sheet.max_row == 1:
        sheet.append(["Description", "Amount", "Date"])

    sheet.append([description, amount, "2024-11-26"])  # Replace with dynamic date if needed
    workbook.save(branch_path)
    print(f"Expense of {amount} for '{description}' recorded in branch '{branch_name}'.")


def generate_report(branch_name):
    """Generate a PDF report with income, expenses, and profit/loss."""
    branch_path = os.path.join(BRANCHES_DIR, branch_name, "database.xlsx")
    if not os.path.exists(branch_path):
        print(f"Branch '{branch_name}' does not exist.")
        return

    workbook = openpyxl.load_workbook(branch_path)
    fees = workbook["Fees"]
    expenses = workbook["Expenses"]

    total_income = sum(row[1] for row in fees.iter_rows(min_row=2, values_only=True) if row[1] is not None)
    total_expenses = sum(row[1] for row in expenses.iter_rows(min_row=2, values_only=True) if row[1] is not None)
    profit_loss = total_income - total_expenses

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.cell(200, 10, txt=f"Branch: {branch_name}", ln=True, align="C")
    pdf.cell(200, 10, txt="Financial Report", ln=True, align="C")
    pdf.cell(200, 10, txt=f"Total Income: {total_income}", ln=True)
    pdf.cell(200, 10, txt=f"Total Expenses: {total_expenses}", ln=True)
    pdf.cell(200, 10, txt=f"Profit/Loss: {profit_loss}", ln=True)

    report_file = os.path.join(branch_path, "financial_report.pdf")
    pdf.output(report_file)
    print(f"Report generated: {report_file}")


def plot_graph(branch_name):
    """Plot income and expenses as a bar graph."""
    branch_path = os.path.join(BRANCHES_DIR, branch_name, "database.xlsx")
    if not os.path.exists(branch_path):
        print(f"Branch '{branch_name}' does not exist.")
        return

    workbook = openpyxl.load_workbook(branch_path)
    fees = workbook["Fees"]
    expenses = workbook["Expenses"]

    total_income = sum(row[1] for row in fees.iter_rows(min_row=2, values_only=True) if row[1] is not None)
    total_expenses = sum(row[1] for row in expenses.iter_rows(min_row=2, values_only=True) if row[1] is not None)

    labels = ["Income", "Expenses"]
    values = [total_income, total_expenses]

    plt.bar(labels, values, color=["green", "red"])
    plt.title(f"Income vs Expenses for {branch_name}")
    plt.ylabel("Amount")
    plt.show()


# Example Usage
if __name__ == "__main__":
    while True:
        print("\n=== School Management System ===")
        print("1. Create Branch")
        print("2. List Branches")
        print("3. Add Student")
        print("4. Record Fee")
        print("5. Record Expense")
        print("6. Generate Report")
        print("7. Plot Graph")
        print("8. Exit")

        choice = input("Enter your choice: ")

        if choice == "1":
            branch = input("Enter branch name: ")
            create_branch(branch)
        elif choice == "2":
            list_branches()
        elif choice == "3":
            branch = input("Enter branch name: ")
            student_id = input("Enter student ID: ")
            name = input("Enter student name: ")
            grade = input("Enter grade: ")
            add_student(branch, student_id, name, grade)
        elif choice == "4":
            branch = input("Enter branch name: ")
            student_id = input("Enter student ID: ")
            amount = float(input("Enter fee amount: "))
            record_fee(branch, student_id, amount)
        elif choice == "5":
            branch = input("Enter branch name: ")
            description = input("Enter expense description: ")
            amount = float(input("Enter expense amount: "))
            record_expense(branch, description, amount)
        elif choice == "6":
            branch = input("Enter branch name: ")
            generate_report(branch)
        elif choice == "7":
            branch = input("Enter branch name: ")
            plot_graph(branch)
        elif choice == "8":
            print("Exiting...")
            break
        else:
            print("Invalid choice. Please try again.")



=== School Management System ===
1. Create Branch
2. List Branches
3. Add Student
4. Record Fee
5. Record Expense
6. Generate Report
7. Plot Graph
8. Exit
Enter your choice: 1
Enter branch name: Bashabo
Branch 'Bashabo' created successfully.

=== School Management System ===
1. Create Branch
2. List Branches
3. Add Student
4. Record Fee
5. Record Expense
6. Generate Report
7. Plot Graph
8. Exit
Enter your choice: 2
Available branches:
- Bashabo

=== School Management System ===
1. Create Branch
2. List Branches
3. Add Student
4. Record Fee
5. Record Expense
6. Generate Report
7. Plot Graph
8. Exit
Enter your choice: 3
Enter branch name: Bashabo
Enter student ID: 1
Enter student name: Aaraf Bin Rashid
Enter grade: Male
Student Aaraf Bin Rashid added to branch 'Bashabo'.

=== School Management System ===
1. Create Branch
2. List Branches
3. Add Student
4. Record Fee
5. Record Expense
6. Generate Report
7. Plot Graph
8. Exit
Enter your choice: 4
Enter branch name: Bashabo
Enter student I